In [ ]:
import os
import json
import numpy as np
from collections import defaultdict

dataset_name = "drawbench-unique"
base_average_score_dir = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium"
seed_list = [42, 123, 456, 789, 1000]
method_list = ["sd-3-5-medium", "FlowGRPO-PickScore", "GRPO-Guard", "DiffusionNFT", "irl_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all_lr_0.0002_ckpt_3200-dpo_top_512_images_no_anime_colorfulness_pickscore_0.02-hpdv3_all", "irl-top_512_images_pickscore_002-civitai_top_sfw_images-uids_lr_0.0002_ckpt_3200-dpo_top_512_images_pickscore_002-civitai_top_sfw_images-uids"]
ckpt_list = [0, 0, 0, 0, 450, 400]
evaluation_metrics = ["pickscore", "imagereward", "hpsv3", "deqa", "aesthetic", "unifiedreward"]

# 存储每个方法的所有 seed 的分数
all_method_scores = {}

# 遍历每个方法
for method, ckpt in zip(method_list, ckpt_list):
    print(f"\n{'='*80}")
    print(f"Processing method: {method} (ckpt: {ckpt})")
    print(f"{'='*80}")
    
    # 存储当前方法所有 seed 的分数
    method_scores = defaultdict(list)
    
    # 遍历所有 seed
    for seed in seed_list:
        average_score_json_path = os.path.join(
            base_average_score_dir, 
            f"generate_images_seed_{seed}/{dataset_name}/{method}/ckpt-{ckpt}/average_scores.json"
        )
        
        if not os.path.exists(average_score_json_path):
            print(f"  Warning: File not found for seed {seed}: {average_score_json_path}")
            continue
        
        # 读取 average_score.json
        with open(average_score_json_path, 'r') as f:
            average_scores = json.load(f)
        
        print(f"  Seed {seed}:")
        # 收集每个指标的分数
        for metric in evaluation_metrics:
            if metric in average_scores:
                score = average_scores[metric]
                method_scores[metric].append(score)
                print(f"    {metric}: {score:.6f}")
            else:
                print(f"    {metric}: Not found")
    
    # 计算每个指标在所有 seed 上的平均值
    method_average_scores = {}
    print(f"\n  Average scores across all seeds:")
    for metric in evaluation_metrics:
        if metric in method_scores and len(method_scores[metric]) > 0:
            avg_score = np.mean(method_scores[metric])
            method_average_scores[metric] = avg_score
            print(f"    {metric}: {avg_score:.6f} (from {len(method_scores[metric])} seeds)")
        else:
            print(f"    {metric}: No data available")
    
    all_method_scores[method] = method_average_scores

# 打印汇总结果
print(f"\n{'='*80}")
print("SUMMARY: Average scores for each method across all seeds")
print(f"{'='*80}")
print(f"{'Method':<70} " + " ".join([f"{m:>12}" for m in evaluation_metrics]))
print("-" * 80)

for method, scores in all_method_scores.items():
    method_display = method[:68] + ".." if len(method) > 70 else method
    score_str = " ".join([f"{scores.get(m, 0):>12.6f}" if m in scores else f"{'N/A':>12}" for m in evaluation_metrics])
    print(f"{method_display:<70} {score_str}")

# # 保存结果到 JSON 文件
# output_file = os.path.join(base_average_score_dir, "average_scores_summary.json")
# with open(output_file, 'w') as f:
#     json.dump(all_method_scores, f, indent=4)
# print(f"\nResults saved to: {output_file}")